# Silver Layer — Cleaning & Validation

Reads the Bronze table, cleans it, and writes the result to the Silver table.

Cleaning clips negative precipitation to zero, drops duplicate `(CITY, DATE)` pairs, and sorts. Dates not present for every region are then dropped, so the `DATE x CITY` pivot in the Gold layer cannot produce NaN; an assertion guards that the grid is complete before writing.

**Input:** `bronze_weather_india.weather.raw_weather`

**Output:** `silver_weather_india.weather.processed_weather`

Normalization, training-sample construction, and the `silver_data` dict are **not** built here — they live in the Gold notebook, which reads this table directly.

## Validate & Clean

In [ ]:
spark_df = spark.read.table("bronze_weather_india.weather.raw_weather")
df = spark_df.toPandas()
df.columns = [c.upper() for c in df.columns]

df["PRECIPITATION_MM"] = df["PRECIPITATION_MM"].clip(lower=0.0)
df = df.drop_duplicates(subset=["CITY", "DATE"], keep="first")
df = df.sort_values(["CITY", "DATE"]).reset_index(drop=True)

# Guarantee a complete DATE x CITY grid. The gold layer pivots on DATE x CITY;
# any date missing a region leaves a NaN hole that silently poisons z-score
# normalization and training. We drop such dates rather than interpolate:
# in practice the gaps sit at the tail of the range where the ERA5 archive
# has not settled yet, and fabricating values there would silently corrupt
# the most recent history the forecast is seeded from.
n_regions = df["CITY"].nunique()
per_date = df.groupby("DATE")["CITY"].nunique()
incomplete = per_date[per_date < n_regions]

if len(incomplete) > 0:
    dropped_dates = sorted(incomplete.index)
    head = [str(d.date()) for d in dropped_dates[:3]]
    tail = [str(d.date()) for d in dropped_dates[-3:]]
    preview = ", ".join(head) if len(dropped_dates) <= 6 else f"{', '.join(head)}, ..., {', '.join(tail)}"
    print(f"WARNING: {len(incomplete)} date(s) are missing rows for one or more "
          f"of the {n_regions} regions and will be dropped: {preview}")
    df = df[df["DATE"].isin(per_date[per_date == n_regions].index)]

assert df.groupby("DATE")["CITY"].nunique().eq(n_regions).all(), "grid still incomplete"
assert not df[["TEMPERATURE_C", "PRECIPITATION_MM", "WIND_SPEED_KMH"]].isna().any().any(), "NaN in silver data"

print(f"Cleaned: {len(df):,} rows | {df['CITY'].nunique()} regions | "
      f"{df['DATE'].min().date()} -> {df['DATE'].max().date()}")

## Assemble Silver Output

In [ ]:
# -- Spark conversion (commented out for future cluster deployment) ----------
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
silver_sdf = spark.createDataFrame(df)

In [ ]:
# Create silver catalog
silver_catalog = "silver_weather_india"
silver_schema = "weather"
silver_table = "processed_weather"

spark.sql(f"CREATE CATALOG IF NOT EXISTS {silver_catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_catalog}.{silver_schema}")

silver_sdf.write \
	.mode("overwrite") \
	.format("csv") \
    .option("header", "true") \
	.option("sep", ",") \
	.option("quote", '"') \
	.option("escape", "\\") \
	.saveAsTable(f"{silver_catalog}.{silver_schema}.{silver_table}")

In [ ]:
df = spark.read.table("silver_weather_india.weather.processed_weather")
df.show()